# RoboBarista – das Milchkännchen bewegt sich! ☕ 🤖

Heute machen wir den **nächsten Schritt nach unserem Kamera-Vergleich**: Das virtuelle Kännchen fährt eine kleine, vorprogrammierte Gießgeste. **Drei Kameras beobachten es gleichzeitig**; unser AprilTag-3-Detektor und die Mehrkamera-Optimierung berechnen unabhängig von der MuJoCo-Wahrheit seine Position, Neigung und Ausgießerspitze.

**Das bekommst du:** eine animierte GIF-Vorschau der drei echten Kamerabilder, ein Diagramm der gemessenen Ausgießerspitze, eine Tabelle mit Zeitstempeln und einen separaten Validierungsbericht. Die feste Tasse wird aus der ersten verwertbaren Kameraaufnahme als Referenz eingemessen; wenn sie später verdeckt ist, verwenden wir **diese gemessene Referenz** weiter (nicht die MuJoCo-Wahrheit).

**Wichtig:** Die Bewegung ist vorerst **synthetisch vorprogrammiert** – noch **keine echte menschliche Demonstration**, keine Roboter-Gelenksteuerung und keine Flüssigkeitssimulation. Die Ground Truth bleibt für die Kontrolle getrennt und fließt nicht in den Positionsschätzer ein.

Eine GPU brauchst du nicht: **Laufzeit → Laufzeittyp ändern → Hardwarebeschleuniger: Keine** funktioniert (OSMesa/CPU). Danach **Laufzeit → Alle ausführen**.


In [ ]:
# 1. GitHub-Projekt laden (keine ZIP nötig)
import os, subprocess, sys, pathlib
repo = pathlib.Path('/content/robobarista-vision')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Coffee-Tobi/robocasa.git', str(repo)], check=True)
else:
    # Bei erneutem Ausführen aktuelle Verbesserungen von GitHub holen.
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
print('Projekt:', repo)


In [ ]:
# 2. Systemgrafik (CPU/OSMesa) und Python-Pakete installieren
# Ohne GPU benötigt MuJoCo echte OSMesa-Systembibliotheken; pip allein reicht nicht.
import os, subprocess, sys, ctypes.util
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'libosmesa6-dev',
                'libgl1', 'libglx-mesa0', 'libgl1-mesa-dri'], check=True)
osmesa_lib = ctypes.util.find_library('OSMesa')
print('OSMesa-Systembibliothek:', osmesa_lib)
if not osmesa_lib:
    raise RuntimeError('OSMesa fehlt trotz apt-Installation. Bitte die Ausgabe von Schritt 2 teilen.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'requirements.txt'], check=True)

# Eigenständiger Importcheck im richtigen Rendering-Backend (ohne GPU: OSMesa).
backend = 'egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
check_env = os.environ.copy()
check_env['MUJOCO_GL'] = backend
if backend == 'osmesa':
    check_env['PYOPENGL_PLATFORM'] = 'osmesa'
print('MuJoCo-Importtest mit Backend:', backend, flush=True)
subprocess.run([sys.executable, '-c',
                'import mujoco; print("MuJoCo:", mujoco.__version__, "Import OK")'],
               env=check_env, check=True)
import cv2, numpy as np
print('OpenCV:', cv2.__version__, 'NumPy:', np.__version__)


In [ ]:
# 3. Mathematische Tests – vollständige Fehleranzeige statt kryptischem Exit-Code
import subprocess,sys
test = subprocess.run(
    [sys.executable, '-m', 'pytest', '-vv', '-ra', '--tb=short', 'tests'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True, errors='replace', check=False)
print('=== Testergebnisse (stdout) ===', flush=True)
print(test.stdout[-30000:] if test.stdout.strip() else '(leer)', flush=True)
print('=== Zusätzliche Fehlermeldungen (stderr) ===', flush=True)
print(test.stderr[-20000:] if test.stderr.strip() else '(leer)', flush=True)
if test.returncode:
    raise RuntimeError(
        'Mindestens ein Python-Test ist fehlgeschlagen. '
        'Oben steht der genaue Testname unter FAILURES/ERRORS; '
        'bitte diesen Abschnitt teilen. Nicht mit unvalidierten Messungen fortfahren.')
print('Alle Python-Tests bestanden.')


In [ ]:
# 4. Kleine virtuelle Gießgeste + drei Kameraaufnahmen + 6D-Positionsschätzung
import os, sys, subprocess
from pathlib import Path
run_env=os.environ.copy()
run_env['MUJOCO_GL']='egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
if run_env['MUJOCO_GL']=='osmesa':
    run_env['PYOPENGL_PLATFORM']='osmesa'
cmd=[sys.executable,'-u','-m','robobarista.record_motion',
     '--frames','25','--fps','8','--width','960','--height','720',
     '--output','runs/motion_capture']
print('Kamera-Modus:',run_env['MUJOCO_GL'])
print('Bewegung: 25 Bilder, 8 virtuelle Bilder pro Sekunde, 3 Kameras',flush=True)
with subprocess.Popen(cmd,env=run_env,cwd=str(repo),text=True,
                      errors='replace',stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT,bufsize=1) as proc:
    for line in proc.stdout:
        print(line,end='',flush=True)
    exit_code=proc.wait()
print('Programm-Exit-Code:',exit_code)
if exit_code:
    raise RuntimeError('Der Kamerafilm ist fehlgeschlagen. Bitte die Fehlermeldung oben kopieren.')


In [ ]:
# 5. Jetzt die Animation ansehen!
from IPython.display import display, Image
from pathlib import Path
import pandas as pd, json
folder=Path(repo)/'runs/motion_capture'
gif=folder/'preview.gif'
if gif.exists():
    display(Image(filename=str(gif),format='gif'))
else:
    print('Noch kein GIF vorhanden – bitte erst Schritt 4 ausführen.')
for file in ('trajectory.png','validation.png'):
    p=folder/file
    if p.exists():
        print(file)
        display(Image(filename=str(p)))
report=folder/'summary.json'
if report.exists():
    s=json.loads(report.read_text())
    print('Kännchen erkannt:',s['pitcher_valid_frames'],'von',s['frames'],'Frames')
    print('Tasse im aktuellen Bild beobachtet:',s['cup_observed_frames'],'von',s['frames'])
    print('Tassenreferenz aus Kamerabild #:',s['cup_reference_frame'])
    print('Spitze relativ zur fixen Tassenreferenz messbar:',s['relative_valid_frames'],'von',s['frames'])
    print('Maximaler Ausgießerfehler:',s['max_spout_tip_error_mm'],'mm')
    print('Maximaler Fehler Spitze zu Tasse:',s['max_relative_spout_to_cup_error_mm'],'mm')
    print('Maximaler Winkelfehler:',s['max_pitcher_rotation_error_deg'],'Grad')
    print('Qualitätsstatus:',s['status'])
    display(pd.read_csv(folder/'trajectory.csv').head(12))


In [ ]:
# 6. Aufnahmen und Messdaten als ZIP sichern (optional)
# Standard: Animation, Messwerte, Kalibrierdateien, Plots;
# setze include_all_camera_frames=True für alle 75 JPG-Einzelbilder.
from IPython.display import FileLink, display
import zipfile
include_all_camera_frames=False
out_zip=folder/'robobarista_motion_capture.zip'
if (folder/'summary.json').exists():
    with zipfile.ZipFile(out_zip,'w',compression=zipfile.ZIP_DEFLATED,
                         compresslevel=6) as z:
        for f in folder.rglob('*'):
            if not f.is_file() or f==out_zip: continue
            if not include_all_camera_frames and 'frames' in f.relative_to(folder).parts:
                continue
            z.write(f,arcname=str(f.relative_to(folder)))
    display(FileLink(str(out_zip)))
else:
    print('Noch kein Bericht vorhanden. Bitte erst die Bewegung starten.')


## Nächster Schritt

Das ist bereits eine **bewegte Kamera-Demonstration unseres virtuellen Kännchens**. Wenn die Animation funktioniert, können wir als Nächstes die Bahn **relativ zur Tasse auswerten und erstmals mit dem FR5-Greifer nachfahren**.

Für eine echte Latte-Art-Demonstration benötigen wir später eine reale oder sinnvoll nachgebildete Kännchenbewegung samt Geschwindigkeitsprofil und Gießphasen. Die jetzige Bewegung ist nur eine synthetische Funktionsprobe.

[Projektplan](https://github.com/Coffee-Tobi/robocasa/blob/main/planning/ROBOBARISTA_DREI_STUFEN_MVP.md) · [GitHub-Code](https://github.com/Coffee-Tobi/robocasa)